# V10 Late Fusion + Evaluation - Intra-Cell DEM Features

Applies V10 Late Fusion (Ridge OOF) to V2+V4 predictions trained with
intra-cell DEM features, then runs the full benchmark evaluation.

**Prerequisites:** Run these notebooks first:
1. `train_v2_convlstm_intracell_dem.ipynb` (generates V2 predictions)
2. `train_v4_gnn_tat_intracell_dem.ipynb` (generates V4 predictions)

**This notebook:**
1. Loads V2 + V4 predictions (BASIC_D10, BASIC_PCA6)
2. Applies Ridge OOF fusion (same method as baseline V10)
3. Computes ACC, FSS, elevation-stratified metrics
4. Compares BASIC vs BASIC_D10 vs BASIC_PCA6
5. Generates LaTeX tables and figures

**No GPU required.** Runs on CPU (~5 minutes total).

## 1. Setup

In [1]:
import os
import sys
import numpy as np

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/ml_precipitation_prediction'
else:
    # Running locally - find project root
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
    if not PROJECT_ROOT or not os.path.exists(os.path.join(PROJECT_ROOT, '.git')):
        PROJECT_ROOT = r'd:\github.com\ninja-marduk\ml_precipitation_prediction'

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

print(f'Working directory: {os.getcwd()}')
print(f'Colab: {IN_COLAB}')

Working directory: d:\github.com\ninja-marduk\ml_precipitation_prediction
Colab: False


## 2. Load Predictions

Load predictions from both baseline (BASIC) and intra-cell DEM (D10, PCA6) runs.

In [ ]:
# =============================================================================
# PREDICTION PATHS
# =============================================================================
EXPERIMENTS = {
    # Baseline (original BASIC features)
    'BASIC': {
        'v2_pred': 'models/output/V2_Enhanced_Models/map_exports/H12/BASIC/ConvLSTM/predictions.npy',
        'v4_pred': 'models/output/V4_GNN_TAT_Models/map_exports/H12/BASIC/GNN_TAT_GAT/predictions.npy',
        'targets': 'models/output/V2_Enhanced_Models/map_exports/H12/BASIC/ConvLSTM/targets.npy',
    },
    # Intra-cell DEM: BASIC + 10 elevation deciles
    'BASIC_D10': {
        'v2_pred': 'models/output/V2_Enhanced_Models_intracell_dem/map_exports/H12/BASIC_D10/ConvLSTM/predictions.npy',
        'v4_pred': 'models/output/V4_GNN_TAT_Models_intracell_dem/map_exports/H12/BASIC_D10/GNN_TAT_GAT/predictions.npy',
        'targets': 'models/output/V2_Enhanced_Models_intracell_dem/map_exports/H12/BASIC_D10/ConvLSTM/targets.npy',
    },
    # Intra-cell DEM: BASIC + 6 PCA components
    'BASIC_PCA6': {
        'v2_pred': 'models/output/V2_Enhanced_Models_intracell_dem/map_exports/H12/BASIC_PCA6/ConvLSTM/predictions.npy',
        'v4_pred': 'models/output/V4_GNN_TAT_Models_intracell_dem/map_exports/H12/BASIC_PCA6/GNN_TAT_GAT/predictions.npy',
        'targets': 'models/output/V2_Enhanced_Models_intracell_dem/map_exports/H12/BASIC_PCA6/ConvLSTM/targets.npy',
    },
    # Intra-cell DEM: BASIC + 10 deciles + 5 statistics (mean, std, skewness, kurtosis, range)
    'BASIC_D10_STATS': {
        'v2_pred': 'models/output/V2_Enhanced_Models_intracell_dem/map_exports/H12/BASIC_D10_STATS/ConvLSTM/predictions.npy',
        'v4_pred': 'models/output/V4_GNN_TAT_Models_intracell_dem/map_exports/H12/BASIC_D10_STATS/GNN_TAT_GAT/predictions.npy',
        'targets': 'models/output/V2_Enhanced_Models_intracell_dem/map_exports/H12/BASIC_D10_STATS/ConvLSTM/targets.npy',
    },
}

# Check which experiments have predictions available
available = {}
for name, paths in EXPERIMENTS.items():
    v2_ok = os.path.exists(os.path.join(PROJECT_ROOT, paths['v2_pred']))
    v4_ok = os.path.exists(os.path.join(PROJECT_ROOT, paths['v4_pred']))
    tgt_ok = os.path.exists(os.path.join(PROJECT_ROOT, paths['targets']))
    status = 'READY' if (v2_ok and v4_ok and tgt_ok) else 'MISSING'
    available[name] = v2_ok and v4_ok and tgt_ok
    print(f'{name}: {status}  (V2={v2_ok}, V4={v4_ok}, targets={tgt_ok})')

experiments_to_run = [name for name, ok in available.items() if ok]
print(f'\nExperiments ready: {experiments_to_run}')

## 3. V10 Late Fusion (Ridge OOF)

In [3]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

def ridge_fusion_oof(v2_pred, v4_pred, targets, alpha=1.0, n_folds=5, seed=42):
    """Out-of-fold Ridge regression fusion (same as baseline V10)."""
    n_samples = v2_pred.shape[0]
    v2_flat = v2_pred.reshape(n_samples, -1)
    v4_flat = v4_pred.reshape(n_samples, -1)
    y_flat = targets.reshape(n_samples, -1)

    X = np.stack([v2_flat, v4_flat], axis=-1)  # (n, pixels, 2)
    oof_pred = np.zeros_like(y_flat)

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    weights_list = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr = X[train_idx].reshape(-1, 2)
        y_tr = y_flat[train_idx].reshape(-1)
        X_va = X[val_idx].reshape(-1, 2)

        ridge = Ridge(alpha=alpha, fit_intercept=True)
        ridge.fit(X_tr, y_tr)
        oof_pred[val_idx] = ridge.predict(X_va).reshape(len(val_idx), -1)
        weights_list.append((ridge.coef_, ridge.intercept_))

    # Reshape back
    fused = oof_pred.reshape(targets.shape)

    # Average weights
    avg_w = np.mean([w[0] for w in weights_list], axis=0)
    avg_b = np.mean([w[1] for w in weights_list])

    return fused, avg_w, avg_b


def compute_metrics(pred, targets):
    """Compute R2, RMSE, MAE, Bias."""
    p = pred.flatten()
    t = targets.flatten()
    return {
        'R2': r2_score(t, p),
        'RMSE': np.sqrt(mean_squared_error(t, p)),
        'MAE': mean_absolute_error(t, p),
        'Bias': np.mean(p - t),
    }

In [4]:
# =============================================================================
# RUN FUSION FOR ALL AVAILABLE EXPERIMENTS
# =============================================================================
results = {}

for exp_name in experiments_to_run:
    paths = EXPERIMENTS[exp_name]
    print(f'\n{"="*60}')
    print(f'  V10 FUSION: {exp_name}')
    print(f'{"="*60}')

    v2_pred = np.load(os.path.join(PROJECT_ROOT, paths['v2_pred']))
    v4_pred = np.load(os.path.join(PROJECT_ROOT, paths['v4_pred']))
    targets = np.load(os.path.join(PROJECT_ROOT, paths['targets']))

    # Squeeze if needed
    if v2_pred.ndim == 5:
        v2_pred = v2_pred.squeeze(-1)
    if v4_pred.ndim == 5:
        v4_pred = v4_pred.squeeze(-1)
    if targets.ndim == 5:
        targets = targets.squeeze(-1)

    print(f'  V2: {v2_pred.shape}, V4: {v4_pred.shape}, Targets: {targets.shape}')

    # Individual model metrics
    v2_metrics = compute_metrics(v2_pred, targets)
    v4_metrics = compute_metrics(v4_pred, targets)
    print(f'  V2:  R2={v2_metrics["R2"]:.4f}, RMSE={v2_metrics["RMSE"]:.2f}')
    print(f'  V4:  R2={v4_metrics["R2"]:.4f}, RMSE={v4_metrics["RMSE"]:.2f}')

    # V10 Fusion
    fused, weights, bias = ridge_fusion_oof(v2_pred, v4_pred, targets)
    v10_metrics = compute_metrics(fused, targets)
    print(f'  V10: R2={v10_metrics["R2"]:.4f}, RMSE={v10_metrics["RMSE"]:.2f}')
    print(f'  Weights: w_V2={weights[0]:.4f}, w_V4={weights[1]:.4f}, bias={bias:.2f}')

    results[exp_name] = {
        'V2': v2_metrics,
        'V4': v4_metrics,
        'V10': v10_metrics,
        'weights': weights,
        'bias': bias,
        'fused_pred': fused,
        'targets': targets,
    }

    # Save V10 predictions
    v10_dir = os.path.join(PROJECT_ROOT, f'models/output/V10_Late_Fusion_intracell_dem/{exp_name}')
    os.makedirs(v10_dir, exist_ok=True)
    np.save(os.path.join(v10_dir, 'predictions.npy'), fused)
    np.save(os.path.join(v10_dir, 'targets.npy'), targets)
    print(f'  Saved: {v10_dir}')


  V10 FUSION: BASIC
  V2: (33, 12, 61, 65), V4: (33, 12, 61, 65), Targets: (33, 12, 61, 65)
  V2:  R2=0.6287, RMSE=81.05
  V4:  R2=0.5974, RMSE=84.40
  V10: R2=0.6658, RMSE=76.90
  Weights: w_V2=0.4469, w_V4=0.7095, bias=-5.55
  Saved: d:\github.com\ninja-marduk\ml_precipitation_prediction\models/output/V10_Late_Fusion_intracell_dem/BASIC

  V10 FUSION: BASIC_D10
  V2: (33, 12, 61, 65), V4: (33, 12, 61, 65), Targets: (33, 12, 61, 65)
  V2:  R2=0.4533, RMSE=98.35
  V4:  R2=0.5424, RMSE=89.98
  V10: R2=0.6283, RMSE=81.10
  Weights: w_V2=0.2081, w_V4=1.0986, bias=-27.61
  Saved: d:\github.com\ninja-marduk\ml_precipitation_prediction\models/output/V10_Late_Fusion_intracell_dem/BASIC_D10

  V10 FUSION: BASIC_PCA6
  V2: (33, 12, 61, 65), V4: (33, 12, 61, 65), Targets: (33, 12, 61, 65)
  V2:  R2=0.4166, RMSE=101.60
  V4:  R2=0.4526, RMSE=98.41
  V10: R2=0.5956, RMSE=84.59
  Weights: w_V2=0.2547, w_V4=1.2026, bias=-42.42
  Saved: d:\github.com\ninja-marduk\ml_precipitation_prediction\models/o

## 4. Comparison: BASIC vs BASIC_D10 vs BASIC_PCA6

In [5]:
import pandas as pd

# Build comparison table
rows = []
for exp_name, res in results.items():
    for model_name in ['V2', 'V4', 'V10']:
        m = res[model_name]
        rows.append({
            'Features': exp_name,
            'Model': model_name,
            'R2': m['R2'],
            'RMSE': m['RMSE'],
            'MAE': m['MAE'],
            'Bias': m['Bias'],
        })

df = pd.DataFrame(rows)
print('\nComparison Table:')
print(df.to_string(index=False, float_format='%.4f'))

# Save comparison
output_dir = os.path.join(PROJECT_ROOT, 'scripts/benchmark/output')
os.makedirs(output_dir, exist_ok=True)
df.to_csv(os.path.join(output_dir, 'intracell_dem_comparison.csv'), index=False)
print(f'\nSaved: {output_dir}/intracell_dem_comparison.csv')


Comparison Table:
  Features Model     R2     RMSE     MAE     Bias
     BASIC    V2 0.6287  81.0534 58.9063 -10.5009
     BASIC    V4 0.5974  84.4014 59.7414 -28.7946
     BASIC   V10 0.6658  76.9000 56.2957  -0.0157
 BASIC_D10    V2 0.4533  98.3477 78.8418  17.4063
 BASIC_D10    V4 0.5424  89.9798 63.9564 -32.9132
 BASIC_D10   V10 0.6283  81.0986 59.8749  -0.0632
BASIC_PCA6    V2 0.4166 101.5969 79.9200   7.0337
BASIC_PCA6    V4 0.4526  98.4111 71.3678 -40.7821
BASIC_PCA6   V10 0.5956  84.5926 63.4951  -0.0376

Saved: d:\github.com\ninja-marduk\ml_precipitation_prediction\scripts/benchmark/output/intracell_dem_comparison.csv


In [6]:
# =============================================================================
# IMPROVEMENT ANALYSIS
# =============================================================================
if 'BASIC' in results:
    baseline_r2 = results['BASIC']['V10']['R2']
    print(f'Baseline V10 R2: {baseline_r2:.4f}\n')

    for exp_name in ['BASIC_D10', 'BASIC_PCA6', 'BASIC_D10_STATS']:
        if exp_name in results:
            new_r2 = results[exp_name]['V10']['R2']
            delta = new_r2 - baseline_r2
            pct = 100 * delta / baseline_r2
            status = 'IMPROVED' if delta > 0 else 'NO IMPROVEMENT'
            print(f'{exp_name}: R2={new_r2:.4f} (delta={delta:+.4f}, {pct:+.1f}%) [{status}]')
else:
    print('BASIC baseline not available. Run baseline predictions first for comparison.')

Baseline V10 R2: 0.6658

BASIC_D10: R2=0.6283 (delta=-0.0375, -5.6%) [NO IMPROVEMENT]
BASIC_PCA6: R2=0.5956 (delta=-0.0702, -10.5%) [NO IMPROVEMENT]


## 5. Run Benchmark Scripts

Run ACC, FSS, and elevation-stratified metrics on the new predictions.

In [7]:
# Run benchmark scripts with --intracell-dem flag for each available bundle
import subprocess

scripts = [
    ('ACC + FSS', 'scripts/benchmark/14_spatiotemporal_metrics.py'),
    ('Elevation-stratified', 'scripts/benchmark/15_elevation_stratified.py'),
    ('Tables + Figures', 'scripts/benchmark/16_generate_new_metrics_figures.py'),
]

bundles_to_evaluate = [name for name in experiments_to_run if name != 'BASIC']

for bundle in bundles_to_evaluate:
    print(f'\n{"="*60}')
    print(f'  Benchmarks for --intracell-dem --bundle {bundle}')
    print(f'{"="*60}')
    for name, script in scripts:
        path = os.path.join(PROJECT_ROOT, script)
        if os.path.exists(path):
            print(f'Running: {name} ({bundle})...')
            result = subprocess.run(
                [sys.executable, path, '--intracell-dem', '--bundle', bundle],
                cwd=PROJECT_ROOT, capture_output=True, text=True
            )
            if result.returncode == 0:
                print(f'  OK: {name}')
            else:
                print(f'  FAILED: {name}')
                print(result.stderr[-300:])
        else:
            print(f'  Script not found: {script}')

# Also run BASIC baseline for comparison (without --intracell-dem flag)
if 'BASIC' in experiments_to_run:
    print(f'\n{"="*60}')
    print(f'  Benchmarks for BASIC (baseline)')
    print(f'{"="*60}')
    for name, script in scripts:
        path = os.path.join(PROJECT_ROOT, script)
        if os.path.exists(path):
            print(f'Running: {name} (BASIC)...')
            result = subprocess.run(
                [sys.executable, path],
                cwd=PROJECT_ROOT, capture_output=True, text=True
            )
            if result.returncode == 0:
                print(f'  OK: {name}')
            else:
                print(f'  FAILED: {name}')
                print(result.stderr[-300:])

print('\nBenchmark evaluation complete.')


  Benchmarks for --intracell-dem --bundle BASIC_D10
Running: ACC + FSS (BASIC_D10)...
  OK: ACC + FSS
Running: Elevation-stratified (BASIC_D10)...
  OK: Elevation-stratified
Running: Tables + Figures (BASIC_D10)...
  OK: Tables + Figures

  Benchmarks for --intracell-dem --bundle BASIC_PCA6
Running: ACC + FSS (BASIC_PCA6)...
  OK: ACC + FSS
Running: Elevation-stratified (BASIC_PCA6)...
  OK: Elevation-stratified
Running: Tables + Figures (BASIC_PCA6)...
  OK: Tables + Figures

  Benchmarks for BASIC (baseline)
Running: ACC + FSS (BASIC)...
  OK: ACC + FSS
Running: Elevation-stratified (BASIC)...
  OK: Elevation-stratified
Running: Tables + Figures (BASIC)...
  OK: Tables + Figures

Benchmark evaluation complete.
